In [ ]:
!wget ftp://ftp.irisa.fr/local/texmex/corpus/sift.tar.gz
!tar -xzf sift.tar.gz

--2026-06-19 07:14:45--  ftp://ftp.irisa.fr/local/texmex/corpus/sift.tar.gz
           => ‘sift.tar.gz’
Resolving ftp.irisa.fr (ftp.irisa.fr)... 131.254.254.45, 2001:660:7303:254::45
Connecting to ftp.irisa.fr (ftp.irisa.fr)|131.254.254.45|:21... connected.
Logging in as anonymous ... Logged in!
==> SYST ... done.    ==> PWD ... done.
==> TYPE I ... done.  ==> CWD (1) /local/texmex/corpus ... done.
==> SIZE sift.tar.gz ... 168280445
==> PASV ... done.    ==> RETR sift.tar.gz ... done.
Length: 168280445 (160M) (unauthoritative)

sift.tar.gz         100%[===================>] 160.48M  16.8MB/s    in 11s     

2026-06-19 07:14:57 (14.7 MB/s) - ‘sift.tar.gz’ saved [168280445]



In [ ]:
import numpy as np
import time
from pyspark.sql import SparkSession

# ==== 1. ΠΑΡΑΜΕΤΡΟΙ ====
L_VALUES_TO_TEST = [ 10, 20, 30 ,40 , 50 , 60]
EPSILON   = 200.0
K_FUNCS   = 4
W_FACTOR  = 1.5
CHUNK_SIZE = 1000
D_DIM     = 128
REC_LEN   = 4 + D_DIM * 4
R_PATH    = "sift/sift_base.fvecs"
S_PATH    = "sift/sift_query.fvecs"

# ==== 2. Spark ====
spark = SparkSession.builder \
    .appName("LSH_Tune_L") \
    .master("local[*]") \
    .config("spark.driver.memory", "6g") \
    .config("spark.python.worker.memory", "2g") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.kryoserializer.buffer.max", "1g") \
    .config("spark.python.worker.faulthandler.enabled", "true") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("WARN")
print(f"Spark {spark.version} | Πυρήνες: {sc.defaultParallelism}")

# ==== 3. Συναρτήσεις ====
def parse_fvecs(record):
    return np.frombuffer(bytes(record[4:]), dtype=np.float32).copy()

def generate_lsh_model(d, L, K, w, seed=42):
    np.random.seed(seed)
    A = [np.random.randn(K, d).astype(np.float32) for _ in range(L)]
    B = [np.random.uniform(0, w, K).astype(np.float32) for _ in range(L)]
    return A, B

def build_bucket_index(S_dict, A, B, w, L):
    buckets = {}
    for s_id, s_vec in S_dict.items():
        for l in range(L):
            proj = np.dot(A[l], s_vec) + B[l]
            key  = (l, tuple(np.floor(proj / w).astype(np.int32).tolist()))
            buckets.setdefault(key, []).append(s_id)
    return buckets

def _process_chunk(ids, vecs, A, B, w, L, s_bkts, s_vecs, eps_sq):
    M = np.stack(vecs)
    candidates = [set() for _ in range(len(ids))]
    for l in range(L):
        projs   = M @ A[l].T + B[l]
        buckets = np.floor(projs / w).astype(np.int32)
        for i in range(len(ids)):
            key = (l, tuple(buckets[i].tolist()))
            if key in s_bkts:
                candidates[i].update(s_bkts[key])
    for i, r_id in enumerate(ids):
        r = M[i]
        for s_id in candidates[i]:
            diff = r - s_vecs[s_id]
            if float(np.dot(diff, diff)) <= eps_sq:
                yield (r_id, s_id)

def lsh_map_chunked(iterator, A_bc, B_bc, w, L, Sb_bc, Sv_bc, eps):
    A = A_bc.value; B = B_bc.value
    sb = Sb_bc.value; sv = Sv_bc.value
    eps_sq = eps ** 2
    ids, vecs = [], []
    for r_id, r_vec in iterator:
        ids.append(r_id); vecs.append(r_vec)
        if len(ids) >= CHUNK_SIZE:
            yield from _process_chunk(ids, vecs, A, B, w, L, sb, sv, eps_sq)
            ids, vecs = [], []
    if ids:
        yield from _process_chunk(ids, vecs, A, B, w, L, sb, sv, eps_sq)

# ==== 4. Φόρτωση δεδομένων (μία φορά) ====
print("\nΦόρτωση datasets...")
N_PARTS = max(sc.defaultParallelism * 4, 20)

rdd_R_kv = (sc.binaryRecords(R_PATH, REC_LEN)
              .map(parse_fvecs).zipWithIndex()
              .filter(lambda x: x[1] < 100000)
              .map(lambda x: (f"R_{x[1]}", x[0]))
              .repartition(N_PARTS)
              .cache())

S_dict = dict(sc.binaryRecords(S_PATH, REC_LEN)
                .map(parse_fvecs).zipWithIndex()
                .filter(lambda x: x[1] < 1000)
                .map(lambda x: (f"S_{x[1]}", x[0]))
                .collect())

R_samp = dict(rdd_R_kv.collect())
S_samp = S_dict
eps_sq_gt = EPSILON ** 2

print(f"  R: {len(R_samp):,} | S: {len(S_samp):,}")

# ==== 5. Ground Truth (μία φορά για όλα τα L) ====
print("Υπολογισμός Ground Truth (εφάπαξ)...")
t_gt = time.time()
gt_pairs = set()
for r_id, r_vec in R_samp.items():
    for s_id, s_vec in S_samp.items():
        d = r_vec - s_vec
        if float(np.dot(d, d)) <= eps_sq_gt:
            gt_pairs.add((r_id, s_id))
print(f"  Ground truth: {len(gt_pairs):,} ζευγάρια σε {time.time()-t_gt:.1f}s")
print(f"  Baseline χρόνος (1000×1000): {time.time()-t_gt:.2f}s")

# ==== 6. Loop πάνω στα L values ====
W_WIDTH = EPSILON * W_FACTOR
results = []

for L in L_VALUES_TO_TEST:
    print(f"\n{'='*55}")
    print(f"  L={L} | K={K_FUNCS} | w={W_FACTOR}×ε={W_WIDTH} | ε={EPSILON}")
    print(f"{'='*55}")

    A_mat, B_mat = generate_lsh_model(D_DIM, L, K_FUNCS, W_WIDTH)
    sb_local     = build_bucket_index(S_dict, A_mat, B_mat, W_WIDTH, L)

    # Closure fix: περνάμε τιμές ως default arguments
    A_bc  = sc.broadcast(A_mat)
    B_bc  = sc.broadcast(B_mat)
    Sv_bc = sc.broadcast(S_dict)
    Sb_bc = sc.broadcast(sb_local)

    t0 = time.time()
    _L, _w, _e = L, W_WIDTH, EPSILON   # τοπικές μεταβλητές για ασφαλές closure
    matches = (rdd_R_kv
               .mapPartitions(lambda it,
                    _A=A_bc, _B=B_bc, _Sb=Sb_bc, _Sv=Sv_bc,
                    _l=_L, _ww=_w, _ee=_e:
                    lsh_map_chunked(it, _A, _B, _ww, _l, _Sb, _Sv, _ee))
               .collect())
    elapsed = time.time() - t0

    ms     = set(matches)
    approx = {p for p in ms if p[0] in R_samp and p[1] in S_samp}
    recall = len(gt_pairs & approx) / len(gt_pairs) if gt_pairs else 1.0
    status = "✓" if recall >= 0.95 else ("~" if recall >= 0.90 else "✗")

    print(f"  Ζευγάρια: {len(matches):,} | Χρόνος: {elapsed:.2f}s | "
          f"Recall: {recall*100:.1f}% {status}")

    results.append((L, elapsed, recall, len(matches)))

    with open("results_L_tuning.txt", "a") as f:
        f.write(f"L={L:>3} K={K_FUNCS} w={W_FACTOR}x eps={EPSILON} | "
                f"time={elapsed:.2f}s recall={recall*100:.1f}% pairs={len(matches)}\n")

    A_bc.destroy(); B_bc.destroy(); Sv_bc.destroy(); Sb_bc.destroy()

# ==== 7. Σύνοψη ====
print(f"\n{'='*55}")
print(f"  {'L':>4} | {'Χρόνος(s)':>10} | {'Recall':>8} | {'Ζευγάρια':>10}")
print(f"  {'-'*46}")
for L, t, rec, pairs in results:
    flag = "✓" if rec >= 0.95 else ("~" if rec >= 0.90 else "✗")
    print(f"  {L:>4} | {t:>10.2f} | {rec*100:>7.1f}% {flag} | {pairs:>10,}")

best = max(results, key=lambda x: (x[2] >= 0.95, -x[1]))
print(f"\n  Βέλτιστο L: {best[0]} → {best[2]*100:.1f}% recall σε {best[1]:.2f}s")
print(f"  → results_L_tuning.txt")

sc.stop()

Spark 4.0.2 | Πυρήνες: 2

Φόρτωση datasets...
  R: 100,000 | S: 1,000
Υπολογισμός Ground Truth (εφάπαξ)...
  Ground truth: 20,477 ζευγάρια σε 265.4s
  Baseline χρόνος (1000×1000): 265.40s

  L=10 | K=4 | w=1.5×ε=300.0 | ε=200.0
  Ζευγάρια: 11,948 | Χρόνος: 14.81s | Recall: 58.3% ✗

  L=20 | K=4 | w=1.5×ε=300.0 | ε=200.0
  Ζευγάρια: 17,062 | Χρόνος: 20.00s | Recall: 83.3% ✗

  L=30 | K=4 | w=1.5×ε=300.0 | ε=200.0
  Ζευγάρια: 19,092 | Χρόνος: 28.69s | Recall: 93.2% ~

  L=40 | K=4 | w=1.5×ε=300.0 | ε=200.0
  Ζευγάρια: 19,855 | Χρόνος: 37.39s | Recall: 97.0% ✓

  L=50 | K=4 | w=1.5×ε=300.0 | ε=200.0
  Ζευγάρια: 20,243 | Χρόνος: 45.37s | Recall: 98.9% ✓

  L=60 | K=4 | w=1.5×ε=300.0 | ε=200.0
  Ζευγάρια: 20,363 | Χρόνος: 52.71s | Recall: 99.4% ✓

     L |  Χρόνος(s) |   Recall |   Ζευγάρια
  ----------------------------------------------
    10 |      14.81 |    58.3% ✗ |     11,948
    20 |      20.00 |    83.3% ✗ |     17,062
    30 |      28.69 |    93.2% ~ |     19,092
    40 |      37